# MNIST parallel batch effects along PC1
## 1. Purpose / experiment definition
Two disjoint, exactly digit-balanced MNIST batches shift in **opposite directions** along the
first principal component fitted to pooled clean data. For a unit PC1 vector v,
$x_i^\lambda=x_i-\lambda m_{c_i}v$ and $y_j^\lambda=y_j+\lambda m_{d_j}v$.
Constant mode uses m=1; heterogeneous mode samples one signed multiplier per digit from Uniform[-2,2].
The same digit multiplier is shared across batches and held fixed across λ for each corruption seed.
The relative constant-mode shift is 2λ. We never center the multiplier vector or clip analytical data.

Both modes run the same EOT/kNN, embedding and evaluation experiments. Figures compare both modes;
MultiMAP additionally compares centered and uncentered preprocessing. This is a new experiment,
not a replacement for the orthogonal paper figure. Scientific code stays visible in cells.

## 2. Imports and global configuration
Configure construction, corruption, OT, competitors, embedding, evaluation and execution here.
`SMOKE=True` reduces observations, seeds and OT/t-SNE iterations, but leaves MultiMAP optimization
at its native defaults. Loading requires the exact same named configuration; no latest-run selection.
Choose the project environment; MultiMAP runs through its existing isolated interpreter.

In [ ]:
from pathlib import Path
from dataclasses import replace, asdict
from functools import partial
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.decomposition import PCA
from torchvision.datasets import MNIST

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
from ne_aligned_utils import paper
from ne_aligned_utils.experiments.synthetic import (
    ExperimentPaths,
    OtConfig,
    EmbeddingConfig,
    EvaluationConfig,
    label_methods,
)
from ne_aligned_utils.experiments.parallel import (
    ParallelConfig,
    ParallelDataConfig,
    ParallelCorruptionConfig,
    ParallelCompetitorsConfig,
    ParallelExperiment,
)
from ne_aligned_utils.evaluation.metrics import compute_kbet, compute_neighbor_purity
from ne_aligned_utils.methods import (
    compute_ot_tsne_embedding_from_plan,
    compute_scanorama_then_tsne_embedding_from_arrays,
    compute_tsne_embedding_from_arrays,
)
from ne_aligned_utils.methods.external.multimap import compute_multimap_embedding_from_arrays
from ne_aligned_utils.paper import layout, registry
from ne_aligned_utils.paper.figures import synthetic as drawing
from ne_aligned_utils.paper.figures import parallel_synthetic as parallel_drawing

paper.use_paper_style()

In [ ]:
RUN_EXPERIMENT = True
RUN_NAME = None
SMOKE = False
SAVE_PAPER_FIGURE = False
RESULTS_ROOT = ROOT / "results"
SOURCE_ROOT = ROOT
if not any((SOURCE_ROOT / rel / "MNIST/raw").is_dir() for rel in ("data/torchvision", "data")):
    SOURCE_ROOT = ROOT.parent / "ne-aligned"
config = ParallelConfig(
    data=ParallelDataConfig(random_seed=0, n_per_digit=100, modes=("constant", "heterogeneous")),
    corruption=ParallelCorruptionConfig(
        magnitudes=(1.0, 4.0, 10.0, 20.0),
        corruption_seeds=(0, 1, 2),
        multiplier_range=(-2.0, 2.0),
        batch_multipliers=(-1.0, 1.0),
    ),
    ot=OtConfig(
        epsilon_policies=("fixed_at_lambda_0", "adaptive_per_magnitude"),
        rel_reg=(1.0, 10.0),
        ot_effective_k=15,
        ot_tol=1e-12,
        ot_max_iter=10_000,
    ),
    competitors=ParallelCompetitorsConfig(
        cross_batch_k=15,
        multimap_profile="default",
        multimap_preprocessing=("centered", "uncentered"),
        run_multimap=True,
        run_scanorama_50_tsne=True,
    ),
    # t-SNE schedule: 250 exaggerated + 500 iterations (openTSNE's defaults, as in the benchmark).
    # To reproduce or load runs saved before this change: n_epochs=200, early_exaggeration_iter=None.
    embedding=EmbeddingConfig(
        embedding_seeds=(0, 1, 2), n_epochs=500, early_exaggeration_iter=250
    ),
    evaluation=EvaluationConfig(kbet_k=15),
)
if SMOKE:
    config = replace(
        config,
        data=replace(config.data, n_per_digit=10),
        corruption=replace(config.corruption, magnitudes=(1.0, 10.0), corruption_seeds=(0,)),
        ot=replace(config.ot, rel_reg=(1.0,)),
        embedding=replace(config.embedding, embedding_seeds=(0,), n_epochs=10),
    )
plot_config = parallel_drawing.ParallelPlotConfig(
    magnitude=10.0, corruption_seed=0, embedding_seed=0, rel_reg=1.0
)
if (
    plot_config.magnitude not in config.corruption.magnitudes
    or plot_config.corruption_seed not in config.corruption.corruption_seeds
    or plot_config.embedding_seed not in config.embedding.embedding_seeds
    or plot_config.rel_reg not in config.ot.rel_reg
):
    raise ValueError("Choose gallery λ/seeds/rel_reg present in the experiment grid")
mnist_root = SOURCE_ROOT / "data/torchvision"
if not (mnist_root / "MNIST/raw").is_dir() and (SOURCE_ROOT / "data/MNIST/raw").is_dir():
    mnist_root = SOURCE_ROOT / "data"
paths = ExperimentPaths(RESULTS_ROOT, mnist_root, run_name=RUN_NAME, download_mnist=False)
experiment = ParallelExperiment(config, paths)
print("Experiment: MNIST parallel PC1 batch effect · constant and heterogeneous")
print(
    f"Batch signs: {config.corruption.batch_multipliers}; heterogeneous range: {config.corruption.multiplier_range}"
)
print(f"Run: {experiment.run_dir.name}\nResults: {experiment.run_dir}")
print(f"MNIST: {mnist_root} · MultiMAP: {config.competitors.multimap_preprocessing}")
if not RUN_EXPERIMENT:
    runs = experiment.load_saved()

## 3. Construct the clean MNIST batches
Sample 2n distinct source indices per digit, split between batches, then shuffle each batch.
The sampling code is identical to the orthogonal walkthrough. Assertions verify disjointness
and exact balance in compute and load modes.

In [ ]:
if RUN_EXPERIMENT:
    dataset = MNIST(paths.data_root, train=True, download=paths.download_mnist)
    all_x = dataset.data.numpy().reshape(-1, 28 * 28)
    all_y = dataset.targets.numpy().astype(int)
    rng = np.random.default_rng(config.data.random_seed)
    batch0_indices, batch1_indices = ([], [])
    for digit in range(10):
        candidates = np.flatnonzero(all_y == digit)
        selected = rng.choice(candidates, 2 * config.data.n_per_digit, replace=False)
        rng.shuffle(selected)
        batch0_indices.extend(selected[: config.data.n_per_digit])
        batch1_indices.extend(selected[config.data.n_per_digit :])
    batch0_indices = np.asarray(batch0_indices)
    batch1_indices = np.asarray(batch1_indices)
    rng.shuffle(batch0_indices)
    rng.shuffle(batch1_indices)
    if np.intersect1d(batch0_indices, batch1_indices).size:
        raise AssertionError("MNIST batches must be disjoint")
    x0_images = all_x[batch0_indices].astype(np.float64) / 255.0
    x1_images = all_x[batch1_indices].astype(np.float64) / 255.0
    labels0 = all_y[batch0_indices]
    labels1 = all_y[batch1_indices]
    expected = np.full(10, config.data.n_per_digit)
    if not (
        np.array_equal(np.bincount(labels0, minlength=10), expected)
        and np.array_equal(np.bincount(labels1, minlength=10), expected)
    ):
        raise AssertionError(
            "Each batch must contain exactly config.data.n_per_digit images per class"
        )
    for name in (
        "x0_images",
        "x1_images",
        "labels0",
        "labels1",
        "batch0_indices",
        "batch1_indices",
    ):
        setattr(experiment, name, locals()[name])
    experiment.labels = np.concatenate([labels0, labels1])
else:
    with np.load(experiment.run_dir / "data/construction.npz", allow_pickle=False) as data:
        labels0, labels1 = data["labels0"], data["labels1"]
        batch0_indices, batch1_indices = data["batch0_indices"], data["batch1_indices"]
assert not np.intersect1d(batch0_indices, batch1_indices).size
counts = pd.DataFrame(
    {"batch 0": np.bincount(labels0, minlength=10), "batch 1": np.bincount(labels1, minlength=10)}
)
assert counts.eq(config.data.n_per_digit).all().all()
print(f"Each batch: {len(labels0)} observations × 784 pixels; distinct source indices.")
display(counts.T)
if not RUN_EXPERIMENT:
    with np.load(experiment.run_dir / "data/construction.npz", allow_pickle=False) as data:
        for name, key in (
            ("x0_images", "clean0"),
            ("x1_images", "clean1"),
            ("labels0", "labels0"),
            ("labels1", "labels1"),
            ("batch0_indices", "batch0_indices"),
            ("batch1_indices", "batch1_indices"),
        ):
            setattr(experiment, name, data[key])
    experiment.labels = np.r_[labels0, labels1]

## 4. Fit PCA and select the parallel direction
Fit full-SVD PCA to pooled clean observations. Use its first unit component, orienting the largest
absolute loading positively so the sign is deterministic. λ is measured in pixel-space L2 units;
PC1 is not multiplied by its eigenvalue or score standard deviation.
The PCA spectrum highlights PC1; the image shows its signed loading pattern.

In [ ]:
if RUN_EXPERIMENT:
    experiment.pca = PCA(n_components=None, svd_solver="full").fit(
        np.vstack([experiment.x0_images, experiment.x1_images])
    )
    experiment.pc1 = experiment.pca.components_[0].copy()
    experiment.pc1 *= np.sign(experiment.pc1[np.argmax(np.abs(experiment.pc1))])
    spectrum = experiment.pca.explained_variance_
else:
    with np.load(experiment.run_dir / "data/construction.npz", allow_pickle=False) as data:
        experiment.pc1, spectrum = data["pc1"], data["pca_variance"]
assert np.isclose(np.linalg.norm(experiment.pc1), 1.0)
print(f"PC1 explains {spectrum[0] / spectrum.sum():.1%} of clean variance")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=layout.full_width(2.0), layout="constrained")
axes[0].semilogy(
    np.arange(len(spectrum)), spectrum / spectrum.sum(), color=registry.resolve("joint-pca").color
)
axes[0].scatter(
    [0], [spectrum[0] / spectrum.sum()], color=registry.resolve("ot").color, label="Selected PC1"
)
axes[0].set(xlabel="PC index (zero-based)", ylabel="Explained variance ratio")
axes[0].legend()
limit = np.max(np.abs(experiment.pc1))
image = axes[1].imshow(experiment.pc1.reshape(28, 28), cmap="RdBu_r", vmin=-limit, vmax=limit)
axes[1].set(title="PC1 direction", xticks=[], yticks=[])
fig.colorbar(image, ax=axes[1], label="Loading")
plt.show()

## 5. Construct constant and heterogeneous shifts
Constant mode uses ten ones. Heterogeneous mode draws one signed multiplier per digit and corruption
seed from Uniform[-2,2], then reuses that draw at every λ. This symmetric range gives both forward
and backward class shifts with typical magnitude similar to the constant multiplier 1.
Changing the range once in configuration changes the run fingerprint. No multiplier centering occurs.

In [ ]:
if RUN_EXPERIMENT:
    experiment.class_multipliers = {}
    audit_records = []
    for seed in config.corruption.corruption_seeds:
        heterogeneous = np.random.default_rng(seed).uniform(
            *config.corruption.multiplier_range, size=10
        )
        for mode in config.data.modes:
            weights = np.ones(10) if mode == "constant" else heterogeneous.copy()
            experiment.class_multipliers[mode, seed] = weights
            for batch, sign in enumerate(config.corruption.batch_multipliers):
                for digit, multiplier in enumerate(weights):
                    audit_records.append(
                        dict(
                            mode=mode,
                            corruption_seed=seed,
                            batch=batch,
                            digit=digit,
                            class_multiplier=multiplier,
                            batch_sign=sign,
                            pc1_coefficient=sign * multiplier,
                        )
                    )
    experiment.audit = pd.DataFrame(audit_records)
    experiment.initialize_run()
    audit = experiment.audit
else:
    audit = pd.concat(
        [
            pd.read_csv(directory / "diagnostics/corruption_audit.csv")
            for directory in runs.run_dirs.values()
        ],
        ignore_index=True,
    )
    experiment.class_multipliers = {
        (mode, seed): part.query("batch == 0").sort_values("digit").class_multiplier.to_numpy()
        for (mode, seed), part in audit.groupby(["mode", "corruption_seed"])
    }
constant = audit.query("mode == 'constant'")
assert constant.class_multiplier.eq(1).all()
lo, hi = config.corruption.multiplier_range
assert audit.query("mode == 'heterogeneous'").class_multiplier.between(lo, hi).all()
display(audit.groupby(["mode", "batch"])["pc1_coefficient"].agg(["min", "max"]).round(3))

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=layout.full_width(2.4), layout="constrained")
for ax, mode in zip(axes, config.data.modes):
    weights = experiment.class_multipliers[mode, plot_config.corruption_seed]
    ax.bar(np.arange(10), weights, color=drawing.GALLERY_BATCH_COLOURS[0])
    ax.axhline(0, color=registry.resolve("joint-pca").color, linewidth=0.7)
    ax.set(
        title=f"{mode.capitalize()} · class multipliers",
        xlabel="Digit",
        ylabel="Multiplier",
        xticks=range(10),
    )
plt.show()

### All digits, both noise modes
Each mode gets its own clean/noisy/clean/noisy grid. Both clean rows show distinct observations;
within each batch the same observation appears before and after its shift. Clipping is display-only.
Change `plot_config.magnitude` to inspect another λ without rerunning an experiment.

In [ ]:
for mode in config.data.modes:
    noisy = experiment.make_batches(mode, plot_config.magnitude, plot_config.corruption_seed)
    representatives = [
        np.array([np.flatnonzero(labels == digit)[0] for digit in range(10)])
        for labels in (experiment.labels0, experiment.labels1)
    ]
    rows = (
        experiment.x0_images[representatives[0]],
        noisy[0][representatives[0]],
        experiment.x1_images[representatives[1]],
        noisy[1][representatives[1]],
    )
    fig, axes = plt.subplots(4, 10, figsize=layout.full_width(2.8), layout="none")
    for row, (examples, name) in enumerate(
        zip(rows, ("Batch 0 clean", "Batch 0 shifted", "Batch 1 clean", "Batch 1 shifted"))
    ):
        for digit, ax in enumerate(axes[row]):
            ax.imshow(np.clip(examples[digit], 0, 1).reshape(28, 28), cmap="gray", vmin=0, vmax=1)
            ax.set(xticks=[], yticks=[])
            for spine in ax.spines.values():
                spine.set_visible(False)
            if row == 0:
                ax.set_title(str(digit))
        axes[row, 0].set_ylabel(name, rotation=0, ha="right", va="center")
    fig.subplots_adjust(left=0.20, right=0.99, bottom=0.03, top=0.89, wspace=0.05, hspace=0.08)
    print(f"{mode.capitalize()} · λ={plot_config.magnitude:g}")
    plt.show()

## 6. Sanity checks / cost decomposition and ε policies
For clean difference d and unit-strength shifts a and b, the cost is
$C^\lambda=C^0+2\lambda d^T(a-b)+\lambda^2\|a-b\|^2$.
Unlike orthogonal noise, the linear interaction generally survives. In constant mode the total
cost change still separates into row and column terms, so balanced OT at fixed ε is invariant.
Heterogeneous class shifts generally introduce nonseparable terms. Adaptive ε is recomputed from
the corrupted costs; fixed ε uses each rel_reg's λ=0 reference.

In [ ]:
def parallel_cost_terms(mode, seed, magnitude):
    base0, base1 = experiment.x0_images, experiment.x1_images
    weights = experiment.class_multipliers[mode, seed]
    signs = config.corruption.batch_multipliers
    a = signs[0] * weights[experiment.labels0, None] * experiment.pc1
    b = signs[1] * weights[experiment.labels1, None] * experiment.pc1
    linear = (
        2
        * magnitude
        * (
            np.sum(base0 * a, axis=1)[:, None]
            - a @ base1.T
            - base0 @ b.T
            + np.sum(base1 * b, axis=1)[None, :]
        )
    )
    quadratic = magnitude**2 * experiment.squared_cost(a, b)
    x0, x1 = experiment.make_batches(mode, magnitude, seed)
    clean_cost = experiment.squared_cost(base0, base1)
    observed = experiment.squared_cost(x0, x1)
    error = np.max(np.abs(observed - clean_cost - linear - quadratic))
    change = observed - clean_cost
    interaction = change - change.mean(axis=0) - change.mean(axis=1)[:, None] + change.mean()
    assert error < 1e-8, "Parallel cost decomposition failed"
    if mode == "constant":
        assert np.max(np.abs(interaction)) < 1e-8, "Constant translation must be separable"
        for base, shifted in ((base0, x0), (base1, x1)):
            np.testing.assert_allclose(
                experiment.squared_cost(base, base),
                experiment.squared_cost(shifted, shifted),
                atol=1e-8,
                rtol=1e-10,
            )
    return x0, x1, error, float(np.sqrt(np.mean(interaction**2)))

In [ ]:
if RUN_EXPERIMENT:
    experiment.reference_plans = {}
    records = []
    for mode in config.data.modes:
        for rel_reg in config.ot.rel_reg:
            for policy in config.ot.epsilon_policies:
                experiment.reference_plans[mode, rel_reg, policy] = experiment.compute_policy_plan(
                    experiment.x0_images, experiment.x1_images, mode, rel_reg, policy
                )
        for seed in config.corruption.corruption_seeds:
            for magnitude in tuple(dict.fromkeys((0.0, *config.corruption.magnitudes))):
                x0, x1, error, interaction = parallel_cost_terms(mode, seed, magnitude)
                for rel_reg in config.ot.rel_reg:
                    for policy in config.ot.epsilon_policies:
                        plan, artifacts = experiment.compute_policy_plan(
                            x0, x1, mode, rel_reg, policy, return_artifacts=True
                        )
                        reference = experiment.reference_plans[mode, rel_reg, policy]
                        records.append(
                            dict(
                                mode=mode,
                                corruption_seed=seed,
                                magnitude=magnitude,
                                rel_reg=rel_reg,
                                epsilon_policy=policy,
                                epsilon=float(artifacts["epsilon"]),
                                epsilon_over_reference=float(artifacts["epsilon"])
                                / experiment.reference_epsilons[mode][rel_reg],
                                relative_ot_plan_change=float(
                                    np.linalg.norm(plan - reference) / np.linalg.norm(reference)
                                ),
                                decomposition_identity_error_max_abs=error,
                                nonseparable_cost_rms=interaction,
                            )
                        )
    experiment.cost_diagnostics = pd.DataFrame(records)
    experiment.save_stage("cost")
    cost = experiment.cost_diagnostics
else:
    cost = runs.cost
control = cost.query("mode == 'constant' and epsilon_policy == 'fixed_at_lambda_0'")
assert len(control) and control.relative_ot_plan_change.max() < 1e-7
assert cost.decomposition_identity_error_max_abs.max() < 1e-8
display(
    cost.groupby(["mode", "epsilon_policy"])[["relative_ot_plan_change", "nonseparable_cost_rms"]]
    .max()
    .round(6)
)

In [ ]:
parallel_drawing.policy_figure(cost, config.data.modes, plot_config.rel_reg)
plt.show()

## 7. Cross-batch affinity experiment: EOT, shared and mutual kNN
Use the same graph diagnostics as the orthogonal experiment. EOT top-k row/column supports,
their union/intersection, shared/mutual kNN purity and nonempty-row coverage are saved for both
noise modes, all corruption seeds, ε policies, rel_reg values and λ (including zero).
Constant translations preserve within-batch neighbors, but can alter cross-batch kNN matching.

In [ ]:
if RUN_EXPERIMENT:
    DIAGNOSTIC_MAGNITUDES = tuple(dict.fromkeys((0.0, *experiment.config.corruption.magnitudes)))
    affinity_records, knn_records = ([], [])
    for mode in experiment.config.data.modes:
        base0, base1 = experiment.clean_by_mode[mode]
        same_digit = experiment.labels0[:, None] == experiment.labels1[None, :]
        for corruption_seed in experiment.config.corruption.corruption_seeds:
            for magnitude in DIAGNOSTIC_MAGNITUDES:
                x0, x1 = experiment.make_batches(mode, magnitude, corruption_seed)
                for rel_reg in experiment.config.ot.rel_reg:
                    for epsilon_policy in experiment.config.ot.epsilon_policies:
                        plan, artifacts = experiment.compute_policy_plan(
                            x0, x1, mode, rel_reg, epsilon_policy, return_artifacts=True
                        )
                        reference = experiment.reference_plans[mode, rel_reg, epsilon_policy]
                        row_support = experiment.rectangular_topk_support(
                            plan, experiment.config.competitors.cross_batch_k, axis=1
                        )
                        column_support = experiment.rectangular_topk_support(
                            plan, experiment.config.competitors.cross_batch_k, axis=0
                        )
                        shared_support = row_support | column_support
                        mutual_support = row_support & column_support
                        row_purity, row_coverage, row_degree = experiment.support_stats(
                            row_support, experiment.labels0, experiment.labels1
                        )
                        column_purity, column_coverage, column_degree = experiment.support_stats(
                            column_support.T, experiment.labels1, experiment.labels0
                        )
                        shared_purity, shared_coverage, shared_degree = experiment.support_stats(
                            shared_support, experiment.labels0, experiment.labels1
                        )
                        mutual_purity, mutual_coverage, mutual_degree = experiment.support_stats(
                            mutual_support, experiment.labels0, experiment.labels1
                        )
                        affinity_records.append(
                            {
                                "mode": mode,
                                "corruption_seed": corruption_seed,
                                "magnitude": magnitude,
                                "rel_reg": rel_reg,
                                "epsilon_policy": epsilon_policy,
                                "epsilon": float(artifacts["epsilon"]),
                                "raw_same_digit_mass_fraction": float(
                                    plan[same_digit].sum() / plan.sum()
                                ),
                                "relative_ot_plan_change": float(
                                    np.linalg.norm(plan - reference) / np.linalg.norm(reference)
                                ),
                                "top_k": experiment.config.competitors.cross_batch_k,
                                "eot_row_digit_purity": row_purity,
                                "eot_row_nonempty_row_fraction": row_coverage,
                                "eot_row_mean_degree": row_degree,
                                "eot_column_digit_purity": column_purity,
                                "eot_column_nonempty_row_fraction": column_coverage,
                                "eot_column_mean_degree": column_degree,
                                "eot_shared_or_digit_purity": shared_purity,
                                "eot_shared_or_nonempty_row_fraction": shared_coverage,
                                "eot_shared_or_mean_degree": shared_degree,
                                "eot_mutual_and_digit_purity": mutual_purity,
                                "eot_mutual_and_nonempty_row_fraction": mutual_coverage,
                                "eot_mutual_and_mean_degree": mutual_degree,
                            }
                        )
                diagnostics = experiment.knn_diagnostics(
                    base0, base1, x0, x1, experiment.config.competitors.cross_batch_k
                )
                knn_records.append(
                    {
                        "mode": mode,
                        "corruption_seed": corruption_seed,
                        "magnitude": magnitude,
                        "k": experiment.config.competitors.cross_batch_k,
                        **diagnostics,
                    }
                )
    experiment.affinity_diagnostics = pd.DataFrame(affinity_records)
    experiment.knn_diagnostics_table = pd.DataFrame(knn_records)
    experiment.save_stage("affinity")
    affinity, knn = experiment.affinity_diagnostics, experiment.knn_diagnostics_table
else:
    affinity, knn = runs.affinity, runs.knn

In [ ]:
parallel_drawing.affinity_figure(affinity, knn, config.data.modes, plot_config.rel_reg)
plt.show()
display(
    knn.groupby(["mode", "magnitude"])[
        [
            "shared_correct_class_neighbor_fraction",
            "mutual_correct_class_neighbor_fraction",
            "mutual_nonempty_row_fraction",
            "within_batch_knn_edge_preservation",
        ]
    ]
    .mean()
    .round(3)
)

### Receiving-side hubness and coverage (cross-batch k=15)
For each receiving point j, count distinct opposite-batch senders: Nₖ(j). Histograms at λ=10
normalize each graph/batch/seed by its own mean, including zero-degree points. The y-axis
is logarithmic to show sparse tails; the x-axis stays linear so anti-hubs at zero remain visible. Tables report
anti-hub fraction, maximum occurrence, and population skewness (undefined for constant counts).
The λ=10 summary averages anti-hub fractions/skewness over seeds and takes the largest
per-seed maximum. Raw per-seed values are saved. Coverage is the fraction with Nₖ≥1,
with mean ± sample SD over the existing corruption seeds.
Directed 0→1 has receiving batch 1 only; directed 1→0 has receiving batch 0 only.

We use the same input representation and Euclidean neighbors as the purity analysis. Mutual/shared
kNN are the intersection/union of both directed selections. Sparsified OT is the **cross-batch
row-top-15/column-top-15 union used in the affinity purity comparison**, at the diagnostic rel_reg
and adaptive ε. This is not the embedding's four-block effective-k graph. Symmetrized shared kNN
and this OT support guarantee positive degree on both sides: their coverage is structurally one,
although their in-degree distributions can be uneven. Heavy tails or concentration near 1 are
hypotheses to inspect, not imposed results. Independent normalization avoids conflating unequal
batch sizes with hubness; no within-batch edges or affinity weights enter the counts.

`COMPUTE_HUBNESS=True` computes only these graph diagnostics, including from a saved construction;
it does not run embeddings or change seeds/λ values. Otherwise read a compatible saved diagnostic.
MultiMAP connectivities and Scanorama MNN pairs were not retained in the existing artifact
interfaces, so those optional receiving-graph diagnostics are skipped (coordinates are not graphs).

In [ ]:
from ne_aligned_utils.evaluation.hubness import (
    cross_batch_occurrences, occurrence_statistics, summarize_coverage, restore_saved_construction,
)
from ne_aligned_utils.paper.figures.hubness import occurrence_histograms, receiving_coverage_figure
from ne_aligned_utils.methods import compute_ot_plan

COMPUTE_HUBNESS = RUN_EXPERIMENT  # Set True to add diagnostics to an already completed run.
HUBNESS_K, HUBNESS_MAGNITUDE = 15, 10.0
hubness_dir = experiment.run_dir / "diagnostics/hubness"
hubness_settings = dict(schema="cross_batch_topk_union_v1", k=HUBNESS_K,
    rel_reg=plot_config.rel_reg, epsilon_policy="adaptive_per_magnitude",
    histogram_magnitude=HUBNESS_MAGNITUDE,
    construction_sha256=json.loads((experiment.run_dir / "config.json").read_text())["construction_sha256"])
hubness_stats = hubness_points = None

In [ ]:
if COMPUTE_HUBNESS:
    if not RUN_EXPERIMENT:
        restore_saved_construction(experiment)  # Saved indices/PCA/directions; no sampling or fitting.
    hubness_records, histogram_records = [], []
    for mode in config.data.modes:
        for corruption_seed in config.corruption.corruption_seeds:
            for magnitude in tuple(dict.fromkeys((0.0, *config.corruption.magnitudes))):
                x0, x1 = experiment.make_batches(mode, magnitude, corruption_seed)
                plan = compute_ot_plan(x0, x1, rel_reg=plot_config.rel_reg,
                    tol=config.ot.ot_tol, max_iter=config.ot.ot_max_iter)
                points = cross_batch_occurrences(
                    experiment.directed_knn(x0, x1, HUBNESS_K),
                    experiment.directed_knn(x1, x0, HUBNESS_K),
                    experiment.rectangular_topk_support(plan, HUBNESS_K, axis=1),
                    experiment.rectangular_topk_support(plan, HUBNESS_K, axis=0))
                metadata = dict(mode=mode, magnitude=magnitude, corruption_seed=corruption_seed,
                                k=HUBNESS_K, rel_reg=plot_config.rel_reg,
                                epsilon_policy="adaptive_per_magnitude")
                hubness_records.append(occurrence_statistics(points).assign(**metadata))
                if magnitude == HUBNESS_MAGNITUDE:
                    histogram_records.append(points.assign(**metadata))
            print(f"Receiving-graph diagnostics: {mode}, corruption seed {corruption_seed}")
    hubness_stats = pd.concat(hubness_records, ignore_index=True)
    hubness_points = (pd.concat(histogram_records, ignore_index=True) if histogram_records
                      else pd.DataFrame(columns=[*points.columns, *metadata]))
    hubness_dir.mkdir(parents=True, exist_ok=True)
    hubness_stats.to_csv(hubness_dir / "receiving_coverage_metrics_raw.csv", index=False)
    hubness_points.to_csv(hubness_dir / "k_occurrence_lambda10.csv", index=False)
    summarize_coverage(hubness_stats).to_csv(
        hubness_dir / "receiving_coverage_metrics_summary.csv", index=False)
    (hubness_dir / "config.json").write_text(json.dumps(hubness_settings, indent=2, sort_keys=True))
elif (hubness_dir / "config.json").is_file():
    if json.loads((hubness_dir / "config.json").read_text()) != hubness_settings:
        raise ValueError("Saved hubness diagnostic settings differ; explicitly recompute diagnostics.")
    hubness_stats = pd.read_csv(hubness_dir / "receiving_coverage_metrics_raw.csv")
    hubness_points = pd.read_csv(hubness_dir / "k_occurrence_lambda10.csv")
else:
    print("No saved receiving-graph diagnostics. Set COMPUTE_HUBNESS=True above and rerun "
          "this subsection; the existing embeddings and purity results are reused.")

In [ ]:
if hubness_stats is not None:
    if not hubness_points.empty:
        for mode in config.data.modes:
            occurrence_histograms(hubness_points.loc[hubness_points["mode"] == mode],
                                  title=f"{mode} · λ=10 · cross-batch k=15")
            plt.show()
        display(hubness_stats.loc[hubness_stats.magnitude == HUBNESS_MAGNITUDE]
            .groupby(["mode", "graph", "receiving_batch"])
            .agg(anti_hub_fraction=("anti_hub_fraction", "mean"),
                 max_k_occurrence=("max_k_occurrence", "max"), skewness=("skewness", "mean"))
            .round(3))
    else:
        print("λ=10 is absent from this run's grid; no histogram cases were added.")
    receiving_coverage_figure(summarize_coverage(hubness_stats), config.data.modes)
    plt.show()
    print(f"Full per-seed statistics, λ=10 counts, summary and settings: {hubness_dir}")

## 8. Embedding experiment
Reuse the same OT-tSNE and Scanorama+t-SNE definitions, public scoring APIs, seeds and strict cache
sidecars as the orthogonal notebook. Every noise mode runs every enabled method. Purity uses the
current macro-by-label definition; digit-conditioned kBET uses the configured k.

In [ ]:
def score_embedding(meta, embedding, runtime, path, display_method):
    batch_ids = np.r_[np.zeros(len(experiment.labels0), int), np.ones(len(experiment.labels1), int)]
    return {
        **meta,
        "display_method": display_method,
        "purity": compute_neighbor_purity(embedding, experiment.labels),
        "kbet": compute_kbet(embedding, batch_ids, experiment.labels, k=config.evaluation.kbet_k),
        "runtime_seconds": runtime,
        "cache_path": str(path),
    }

### OT-tSNE
Keep both fixed/adaptive ε policies and the configured rel_reg grid. Plans are reused across
embedding seeds within a given mode/corruption/λ case.

In [ ]:
def ot_embeddings(x0, x1, mode, magnitude, corruption_seed, embedding_seed):
    records = []
    for rel_reg in experiment.config.ot.rel_reg:
        for epsilon_policy in experiment.config.ot.epsilon_policies:
            plan_key = (mode, corruption_seed, magnitude, rel_reg, epsilon_policy)
            if plan_key not in embedding_plan_cache:
                embedding_plan_cache[plan_key] = experiment.compute_policy_plan(
                    x0, x1, mode, rel_reg, epsilon_policy, return_artifacts=True
                )
            plan, artifacts = embedding_plan_cache[plan_key]
            display_method = f"OT-tSNE ({epsilon_policy}, rel_reg={rel_reg:g})"
            path = experiment.cache_path(
                mode,
                magnitude,
                "ot",
                "tsne",
                corruption_seed,
                embedding_seed,
                epsilon_policy,
                rel_reg,
                experiment.config.ot.ot_effective_k,
            )
            meta = dict(
                mode=mode,
                magnitude=magnitude,
                method="ot",
                backend="tsne",
                corruption_seed=corruption_seed,
                embedding_seed=embedding_seed,
                rel_reg=rel_reg,
                epsilon=float(artifacts["epsilon"]),
                epsilon_policy=epsilon_policy,
                effective_k=experiment.config.ot.ot_effective_k,
            )
            embedding, runtime = experiment.load_or_compute(
                path,
                lambda plan=plan, embedding_seed=embedding_seed, x0=x0, x1=x1: (
                    compute_ot_tsne_embedding_from_plan(
                        plan,
                        effective_k=experiment.config.ot.ot_effective_k,
                        random_state=embedding_seed,
                        n_epochs=experiment.config.embedding.n_epochs,
                        early_exaggeration_iter=experiment.config.embedding.early_exaggeration_iter,
                        init="random",
                        backend="tsne",
                        x_joint=np.vstack([x0, x1]),
                    )
                ),
                meta,
            )
            records.append(score_embedding(meta, embedding, runtime, path, display_method))
    return records

### Scanorama + t-SNE
Exact Scanorama matching, 50 integrated dimensions, then t-SNE with perplexity 30.

In [ ]:
def scanorama_embeddings(x0, x1, mode, magnitude, corruption_seed, embedding_seed):
    records = []
    if experiment.config.competitors.run_scanorama_50_tsne:
        display_method, key = ("Scanorama + t-SNE (50d)", "scanorama-50-tsne")
        path = experiment.cache_path(mode, magnitude, key, "tsne", corruption_seed, embedding_seed)
        meta = dict(
            mode=mode,
            magnitude=magnitude,
            method=key,
            backend="tsne",
            corruption_seed=corruption_seed,
            embedding_seed=embedding_seed,
            epsilon_policy=None,
            rel_reg=None,
            effective_k=None,
            integration_dim=50,
            perplexity=30.0,
            n_epochs=experiment.config.embedding.n_epochs,
        )
        embedding, runtime = experiment.load_or_compute(
            path,
            lambda embedding_seed=embedding_seed, x0=x0, x1=x1: (
                compute_scanorama_then_tsne_embedding_from_arrays(
                    x0,
                    x1,
                    scanorama_dim=50,
                    random_state=embedding_seed,
                    perplexity=30.0,
                    n_jobs=1,
                    scanorama_kwargs={"approx": False},
                    n_iter=experiment.config.embedding.n_epochs,
                    early_exaggeration_iter=experiment.config.embedding.early_exaggeration_iter,
                    backend="tsne",
                )
            ),
            meta,
        )
        records.append(score_embedding(meta, embedding, runtime, path, display_method))
    return records

### MultiMAP: centered and uncentered
`multimap_profile` still chooses native defaults or k=15. `multimap_preprocessing` defaults to both
centered and uncentered, and each gets a distinct method/cache ID.
Centered uses the wrapper's native scaling and centered PCA. Uncentered uses the existing preset:
no scaling, no per-batch centering, and TruncatedSVD. Thus this comparison includes both centering
and scaling differences; it is not a pure centering-only ablation. Other layout parameters stay
at native defaults. In particular, smoke mode does not override MultiMAP epochs.

In [ ]:
def multimap_embeddings(x0, x1, mode, magnitude, corruption_seed, embedding_seed):
    records = []
    if not config.competitors.run_multimap:
        return records
    k = 15 if config.competitors.multimap_profile == "k15" else None
    for preprocessing in config.competitors.multimap_preprocessing:
        backend = "native-defaults" if preprocessing == "centered" else "uncentered-svd-defaults"
        kwargs = {} if k is None else {"n_neighbors": k}
        if preprocessing == "uncentered":
            kwargs.update(scale=False, center_batches=False, use_native_scaling=False)
        meta = dict(
            mode=mode,
            magnitude=magnitude,
            method="multimap",
            backend=backend,
            corruption_seed=corruption_seed,
            embedding_seed=embedding_seed,
            epsilon_policy=None,
            rel_reg=None,
            effective_k=k,
            preprocessing=preprocessing,
            multimap_profile=config.competitors.multimap_profile,
        )
        path = experiment.cache_path(
            mode, magnitude, "multimap", backend, corruption_seed, embedding_seed, k=k
        )
        embedding, runtime = experiment.load_or_compute(
            path,
            partial(
                compute_multimap_embedding_from_arrays,
                x0,
                x1,
                random_state=embedding_seed,
                **kwargs,
            ),
            meta,
        )
        records.append(score_embedding(meta, embedding, runtime, path, "MultiMAP"))
    return records

In [ ]:
def prepare_joint_gallery():
    for mode in config.data.modes:
        x0, x1 = experiment.make_batches(mode, plot_config.magnitude, plot_config.corruption_seed)
        path = experiment.cache_path(
            mode,
            plot_config.magnitude,
            "joint-tsne",
            "tsne",
            plot_config.corruption_seed,
            plot_config.embedding_seed,
        )
        metadata = dict(
            mode=mode,
            method="joint-tsne",
            magnitude=plot_config.magnitude,
            corruption_seed=plot_config.corruption_seed,
            embedding_seed=plot_config.embedding_seed,
            perplexity=30.0,
            pca_components=50,
            n_iter=None,
        )
        experiment.load_or_compute(
            path,
            partial(
                compute_tsne_embedding_from_arrays,
                x0,
                x1,
                random_state=plot_config.embedding_seed,
                perplexity=30.0,
                pca_components=50,
            ),
            metadata,
        )

In [ ]:
if RUN_EXPERIMENT:
    records, embedding_plan_cache = [], {}
    for mode in config.data.modes:
        for corruption_seed in config.corruption.corruption_seeds:
            for magnitude in config.corruption.magnitudes:
                x0, x1 = experiment.make_batches(mode, magnitude, corruption_seed)
                embedding_plan_cache.clear()
                for embedding_seed in config.embedding.embedding_seeds:
                    for method in (ot_embeddings, scanorama_embeddings, multimap_embeddings):
                        records.extend(
                            method(x0, x1, mode, magnitude, corruption_seed, embedding_seed)
                        )
                print(f"Completed {mode}, corruption seed {corruption_seed}, λ={magnitude:g}")
    experiment.results = label_methods(pd.DataFrame(records))
    experiment.save_stage("embedding")
    # Cache the same joint PCA-50/t-SNE control separately for each noise mode.
    prepare_joint_gallery()
    results = experiment.results
else:
    results = runs.metrics
print(f"{len(results)} per-seed embedding evaluations; complete table saved on disk.")

### Representative embeddings: both modes and both MultiMAP variants
The gallery reads cached coordinates at the chosen λ/seeds. Each mode has a batch-colored row
and a digit-colored row, including joint t-SNE, OT-tSNE and both MultiMAP variants. Choose a gallery
λ/seed/rel_reg that is present in the configured embedding grid.

In [ ]:
parallel_drawing.embedding_gallery(experiment.run_dir, plot_config)
plt.show()

## 9. Aggregate and inspect results
First average embedding seeds within each corruption draw, then summarize across corruption draws.
The identical aggregation code keeps mode, method ID, preprocessing, rel_reg and ε policy explicit.
Both noise modes appear in the purity/kBET plots; solid/dashed MultiMAP curves distinguish
centered/uncentered preprocessing. Constant-mode corruption draws are identical constructions;
its seed variation measures embedding variability, not new noise realizations.

In [ ]:
if RUN_EXPERIMENT:
    if "preprocessing" not in experiment.results:
        experiment.results["preprocessing"] = None
    experiment.per_corruption_summary = (
        experiment.results.groupby(
            [
                "mode",
                "magnitude",
                "method_id",
                "method",
                "display_method",
                "epsilon_policy",
                "rel_reg",
                "effective_k",
                "preprocessing",
                "corruption_seed",
            ],
            dropna=False,
        )
        .agg(
            purity=("purity", "mean"),
            purity_embedding_sd=("purity", "std"),
            kbet=("kbet", "mean"),
            kbet_embedding_sd=("kbet", "std"),
            n_embedding_seeds=("embedding_seed", "size"),
        )
        .reset_index()
    )
    experiment.summary = (
        experiment.per_corruption_summary.groupby(
            [
                "mode",
                "magnitude",
                "method_id",
                "method",
                "display_method",
                "epsilon_policy",
                "rel_reg",
                "effective_k",
                "preprocessing",
            ],
            dropna=False,
        )
        .agg(
            purity=("purity", "mean"),
            purity_std=("purity", "std"),
            purity_embedding_sd_mean=("purity_embedding_sd", "mean"),
            kbet=("kbet", "mean"),
            kbet_std=("kbet", "std"),
            kbet_embedding_sd_mean=("kbet_embedding_sd", "mean"),
            n_corruption_seeds=("corruption_seed", "size"),
        )
        .reset_index()
    )
    experiment.save_stage("summary")
else:
    print("Using saved artifacts for this stage.")

In [ ]:
runs = experiment.load_saved()
parallel_drawing.metrics_figure(runs.summary, config.data.modes, plot_config.rel_reg)
plt.show()
display(
    runs.summary[
        [
            "mode",
            "magnitude",
            "display_method",
            "preprocessing",
            "epsilon_policy",
            "rel_reg",
            "purity",
            "purity_std",
            "kbet",
            "kbet_std",
        ]
    ]
    .head(20)
    .round(3)
)

### Results without fixed ε
Keep adaptive-ε OT and all competitors, with the same colors, axes, and uncertainty summaries
as the full comparison above. The saved results retain both ε policies.

In [ ]:
adaptive_summary = runs.summary.loc[
    ~runs.summary["epsilon_policy"].eq("fixed_at_lambda_0")
]
parallel_drawing.metrics_figure(adaptive_summary, config.data.modes, plot_config.rel_reg)
plt.show()

## 10. Reproduce comparison figures from saved results
`build_parallel_figures()` reloads the explicit run through the strict artifact loader. It draws
the same policy, affinity, embedding and metric figures used above, with **both** noise modes.
The separate parallel builder does not modify or relabel the orthogonal manuscript figure.
`SAVE_PAPER_FIGURE=True` saves under `results/paper_figures/synthetic_experiment/parallel_pc1/<run-name>/`.
Otherwise figures are displayed only. Rendering never computes OT or an embedding.

The hierarchy follows the orthogonal experiment: a root config/construction plus `constant/` and
`heterogeneous/`, each with diagnostics, raw/per-corruption/summary metric CSVs, and per-method
embedding caches with JSON sidecars. Reuse the same config and `RUN_NAME` to reload a completed run.

In [ ]:
output_dir = (
    (ROOT / "results/paper_figures/synthetic_experiment/parallel_pc1" / experiment.run_dir.name)
    if SAVE_PAPER_FIGURE
    else None
)
figures = parallel_drawing.build_parallel_figures(
    experiment.run_dir, output_dir=output_dir, selection=plot_config
)
for name, fig in figures.items():
    print(f"{name}: constant and heterogeneous · {experiment.run_dir.name}")
    display(fig)
    plt.close(fig)